# Hybrid Retrieval Pipeline v4
**BM25 bigram + Dense (ChromaDB) + RRF (ID-corrected) + Reranker + Chunk Expand (word-guarded)**

```
Query → [BM25 bigram top-30] ──(convert chunk_id→chroma_id)──┐
                                                               ├─ RRF → top-10 → Reranker → dedup(1) → top-3
Query → [Dense top-30]       ──(chroma_id native)────────────┘
                                └→ expand: ghép full điều luật + word-count guard
```

## Fixes so với v3

| # | Vấn đề v3 | Fix v4 |
|---|-----------|--------|
| **FIX 1** | BM25 dùng `parquet chunk_id`, ChromaDB dùng `SHA-256` → RRF không bao giờ boost chunk xuất hiện ở cả hai | Build `_bm25_to_chroma` map khi init; convert BM25 IDs sang chroma IDs **trước** khi RRF |
| **FIX 2** | `retrieve_forms()` 1-candidate gọi `col.get()` → lấy chunk đầu tiên ngẫu nhiên, không phải relevant nhất | Thay bằng `col.query()` với embedding → luôn trả chunk semantically gần query nhất |
| **FIX 3** | `expand_legal_chunk()` cần `article` trong ChromaDB metadata nhưng không được kiểm tra | Assert `article` trong parquet columns; spot-check ChromaDB metadata khi init; warning + fallback rõ ràng |
| **FIX 4** | Expand không giới hạn tổng từ → có thể tràn context window LLM | Guard bằng `MAX_EXPAND_WORDS=2400`; truncate thông minh chunk cuối thay vì cắt cứng |

## 0. Cài đặt

In [1]:
%pip install rank-bm25 sentence-transformers chromadb unicodedata2 pandas pyarrow torch python-dotenv huggingface_hub ipywidgets

Note: you may need to restart the kernel to use updated packages.


## 1. Imports & Config

In [2]:
import hashlib
import re
import time
import pickle
import unicodedata
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import torch
import chromadb
from chromadb.config import Settings
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

print("Imports OK")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Imports OK
Device: cpu


In [3]:
# ============================================================
# CONFIG
# ============================================================
PROJECT_ROOT = Path("..").resolve()
CHUNK_DIR    = PROJECT_ROOT / "dataset" / "chunks"
CHROMA_DIR   = PROJECT_ROOT / "dataset" / "chromadb"
BM25_DIR     = PROJECT_ROOT / "dataset" / "bm25"
MODEL_EMBED  = PROJECT_ROOT / "models" / "embedding"
MODEL_RERANK = PROJECT_ROOT / "models" / "reranker"

EMBED_MODEL_NAME  = "intfloat/multilingual-e5-large-instruct"
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

# Retrieval params
BM25_TOP_K         = 30
DENSE_TOP_K        = 30
RRF_K              = 60
RERANK_TOP_N       = 10
FINAL_TOP_K        = 3
MAX_CHUNKS_PER_DOC = 1
MAX_EXPAND_CHUNKS  = 6
MAX_EXPAND_WORDS   = 2400   # FIX 4: ~400 từ/chunk × 6 chunks

INSTRUCTIONS = {
    "legal"   : "Instruct: Tim dieu khoan phap luat Viet Nam lien quan.\nQuery: ",
    "forms"   : "Instruct: Tim mau bieu hanh chinh phu hop voi nhu cau soan thao.\nQuery: ",
    "examples": "Instruct: Tim vi du van ban hanh chinh tuong tu tinh huong sau.\nQuery: ",
}

# Sanity check paths
for name, path in {
    "legal_chunks"   : CHUNK_DIR / "legal_chunks.parquet",
    "forms_chunks"   : CHUNK_DIR / "forms_chunks.parquet",
    "examples_chunks": CHUNK_DIR / "examples_chunks.parquet",
    "chromadb"       : CHROMA_DIR,
}.items():
    print(f"  {'✅' if path.exists() else '❌'} {name}: {path}")

  ✅ legal_chunks: /Users/capkimkhanh/Documents/DUT/QLDA/RagDraftingAI/RAG/dataset/chunks/legal_chunks.parquet
  ✅ forms_chunks: /Users/capkimkhanh/Documents/DUT/QLDA/RagDraftingAI/RAG/dataset/chunks/forms_chunks.parquet
  ✅ examples_chunks: /Users/capkimkhanh/Documents/DUT/QLDA/RagDraftingAI/RAG/dataset/chunks/examples_chunks.parquet
  ✅ chromadb: /Users/capkimkhanh/Documents/DUT/QLDA/RagDraftingAI/RAG/dataset/chromadb


## 2. Text Normalization & Bigram Tokenization

In [4]:
def remove_diacritics(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    stripped = "".join(c for c in nfd if unicodedata.category(c) != "Mn")
    return stripped.replace("đ", "d").replace("Đ", "D")


def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    return re.sub(r"\s+", " ", text)


def tokenize_for_bm25(text: str, use_bigrams: bool = True) -> List[str]:
    text = normalize_text(text)
    text = remove_diacritics(text)
    all_tokens = re.findall(r"[a-z][a-z0-9]*", text)
    unigrams   = [t for t in all_tokens if len(t) >= 2]
    if not use_bigrams:
        return unigrams
    bigrams = [f"{a}_{b}" for a, b in zip(all_tokens, all_tokens[1:])]
    return unigrams + bigrams


# Smoke test
for t in ["Quy định về hòa giải ở cơ sở", "trách nhiệm của Bộ trưởng"]:
    tokens = tokenize_for_bm25(t)
    print(f"Input   : {t}")
    print(f"Unigrams: {[x for x in tokens if '_' not in x]}")
    print(f"Bigrams : {[x for x in tokens if '_' in x]}\n")

Input   : Quy định về hòa giải ở cơ sở
Unigrams: ['quy', 'dinh', 've', 'hoa', 'giai', 'co', 'so']
Bigrams : ['quy_dinh', 'dinh_ve', 've_hoa', 'hoa_giai', 'giai_o', 'o_co', 'co_so']

Input   : trách nhiệm của Bộ trưởng
Unigrams: ['trach', 'nhiem', 'cua', 'bo', 'truong']
Bigrams : ['trach_nhiem', 'nhiem_cua', 'cua_bo', 'bo_truong']



## 3. FIX 1 — ChromaDB ID Map

**Vấn đề v3:** BM25 trả về `parquet chunk_id` (text-key), ChromaDB trả về `SHA-256 hex ID`.
Khi RRF merge hai danh sách, không có ID nào trùng nhau → RRF chỉ là union, không boost được
chunk xuất hiện ở cả hai nguồn.

**Fix:** Tái tạo `chroma_id = SHA-256[:24](doc_id|chunk_index|text[:200])` giống hệt
`embedAndIndex.ipynb`, build bảng ánh xạ `parquet_chunk_id → chroma_id` một lần khi init,
rồi convert BM25 results trước khi đưa vào RRF.

In [5]:
def _make_chroma_id(doc_id: str, chunk_index: int, text: str) -> str:
    """
    Tái tạo ChromaDB ID giống hệt embedAndIndex.ipynb:
        SHA-256[:24] của f"{doc_id}|{chunk_index}|{text[:200]}"
    """
    raw = f"{doc_id}|{chunk_index}|{text[:200]}"
    return hashlib.sha256(raw.encode()).hexdigest()[:24]


def _build_chroma_id_map(df: pd.DataFrame) -> Dict[str, str]:
    """
    FIX 1: Build bảng ánh xạ parquet chunk_id → ChromaDB SHA-256 ID.
    Dùng để convert BM25 results trước khi đưa vào RRF.
    """
    mapping: Dict[str, str] = {}
    for row in df.itertuples(index=False):
        chroma_id = _make_chroma_id(
            str(row.doc_id),
            int(row.chunk_index),
            str(row.text),
        )
        mapping[str(row.chunk_id)] = chroma_id
    return mapping


print("FIX 1 helpers defined.")

FIX 1 helpers defined.


## 4. Form Keyword Pre-filter & Detection

In [6]:
FORM_KEYWORD_MAP: List[Tuple[str, List[str]]] = [
    (r"nghi\s*quyet",                                               ["Form_01"]),
    (r"quyet\s*dinh",                                               ["Form_02", "Form_03"]),
    (r"to\s*trinh|trinh\s*duyet|trinh\s*phe\s*duyet",              ["Form_04"]),
    (r"bao\s*cao",                                                  ["Form_04"]),
    (r"chi\s*thi",                                                  ["Form_04"]),
    (r"thong\s*bao|thong\s*cao",                                    ["Form_04"]),
    (r"huong\s*dan",                                                ["Form_04"]),
    (r"ke\s*hoach|chuong\s*trinh|phuong\s*an|de\s*an|du\s*an",     ["Form_04"]),
    (r"quy\s*che|quy\s*dinh",                                       ["Form_04"]),
    (r"uy\s*quyen|giay\s*uy\s*quyen",                               ["Form_04"]),
    (r"phieu\s*gui|phieu\s*chuyen|phieu\s*bao",                     ["Form_04"]),
    (r"cong\s*van",                                                 ["Form_05"]),
    (r"cong\s*dien|khan\s*cap|thong\s*bao\s*khan",                  ["Form_06"]),
    (r"giay\s*moi|thu\s*moi|moi\s*hop|moi\s*hoi\s*nghi",           ["Form_07"]),
    (r"giay\s*gioi\s*thieu|gioi\s*thieu\s*can\s*bo",                ["Form_08"]),
    (r"bien\s*ban",                                                 ["Form_09"]),
    (r"giay\s*nghi\s*phep|nghi\s*phep|xin\s*nghi",                 ["Form_10"]),
]

_COMPILED_FORM_PATTERNS = [
    (re.compile(pat, re.IGNORECASE), form_ids)
    for pat, form_ids in FORM_KEYWORD_MAP
]


def detect_form_candidates(query: str) -> Optional[List[str]]:
    q_norm = remove_diacritics(normalize_text(query))
    candidates: List[str] = []
    for pattern, form_ids in _COMPILED_FORM_PATTERNS:
        if pattern.search(q_norm):
            for fid in form_ids:
                if fid not in candidates:
                    candidates.append(fid)
    return candidates if candidates else None


# Test
test_cases = [
    ("soạn thảo công văn gửi sở tài chính",   ["Form_05"]),
    ("làm tờ trình đề nghị phê duyệt",        ["Form_04"]),
    ("quyết định bổ nhiệm cán bộ",            ["Form_02", "Form_03"]),
    ("nghị quyết phiên họp thường kỳ",        ["Form_01"]),
    ("biên bản cuộc họp",                     ["Form_09"]),
    ("giấy mời họp ban giám đốc",             ["Form_07"]),
    ("công điện khẩn thiên tai",              ["Form_06"]),
    ("giấy nghỉ phép năm",                    ["Form_10"]),
    ("giấy giới thiệu cán bộ",               ["Form_08"]),
    ("soạn thảo văn bản hành chính",          None),
]
print("Keyword detection test:")
for q, expected in test_cases:
    result = detect_form_candidates(q)
    ok = result == expected
    print(f"  {'✅' if ok else '❌'} '{q}'")
    if not ok:
        print(f"       got={result}  expected={expected}")

Keyword detection test:
  ✅ 'soạn thảo công văn gửi sở tài chính'
  ✅ 'làm tờ trình đề nghị phê duyệt'
  ✅ 'quyết định bổ nhiệm cán bộ'
  ✅ 'nghị quyết phiên họp thường kỳ'
  ✅ 'biên bản cuộc họp'
  ✅ 'giấy mời họp ban giám đốc'
  ✅ 'công điện khẩn thiên tai'
  ✅ 'giấy nghỉ phép năm'
  ✅ 'giấy giới thiệu cán bộ'
  ✅ 'soạn thảo văn bản hành chính'


## 5. Build & Load BM25 Indexes

In [7]:
def build_bm25_index(
    df: pd.DataFrame,
    text_col: str,
    save_path: Path,
    name: str,
    force_rebuild: bool = False,
) -> Tuple[BM25Okapi, List[str]]:
    """Build hoặc load BM25. Trả về (bm25, list_of_parquet_chunk_ids)."""
    meta_path = save_path.with_suffix(".meta.pkl")
    assert "chunk_id" in df.columns, f"[{name}] DataFrame thiếu cột 'chunk_id'"

    if save_path.exists() and meta_path.exists() and not force_rebuild:
        print(f"[{name}] Loading from cache: {save_path}")
        t0 = time.time()
        with open(save_path, "rb") as f:
            bm25 = pickle.load(f)
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        print(f"  Loaded {len(meta['chunk_ids']):,} docs in {time.time()-t0:.1f}s")
        return bm25, meta["chunk_ids"]

    print(f"[{name}] Building BM25 bigram index ({len(df):,} docs)...")
    t0 = time.time()
    chunk_ids     = df["chunk_id"].tolist()
    corpus_tokens = [
        tokenize_for_bm25(text)
        for text in tqdm(df[text_col], desc=f"  Tokenizing {name}", leave=False)
    ]
    bm25 = BM25Okapi(corpus_tokens)
    with open(save_path, "wb") as f:
        pickle.dump(bm25, f, protocol=pickle.HIGHEST_PROTOCOL)
    with open(meta_path, "wb") as f:
        pickle.dump({"chunk_ids": chunk_ids}, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Done in {time.time()-t0:.1f}s | {save_path.stat().st_size/1024/1024:.1f} MB")
    return bm25, chunk_ids


print("BM25 builder defined.")

BM25 builder defined.


In [8]:
FORCE_REBUILD = False

print("Loading chunk files...")
df_legal    = pd.read_parquet(CHUNK_DIR / "legal_chunks.parquet")
df_forms    = pd.read_parquet(CHUNK_DIR / "forms_chunks.parquet")
df_examples = pd.read_parquet(CHUNK_DIR / "examples_chunks.parquet")
print(f"  legal={len(df_legal):,}  forms={len(df_forms)}  examples={len(df_examples)}")

# FIX 3: verify 'article' column
assert "article" in df_legal.columns, \
    "legal_chunks.parquet thiếu cột 'article' — expand sẽ không hoạt động!"
print("✅ 'article' column present in legal_chunks")

# BM25 indexes
BM25_DIR.mkdir(parents=True, exist_ok=True)
bm25_legal,    ids_legal    = build_bm25_index(df_legal,    "text", BM25_DIR / "bm25_legal_v2.pkl",    "legal",    FORCE_REBUILD)
bm25_forms,    ids_forms    = build_bm25_index(df_forms,    "text", BM25_DIR / "bm25_forms_v2.pkl",    "forms",    FORCE_REBUILD)
bm25_examples, ids_examples = build_bm25_index(df_examples, "text", BM25_DIR / "bm25_examples_v2.pkl", "examples", FORCE_REBUILD)

print("\n✅ BM25 indexes ready.")

# Metadata lookups (parquet chunk_id → row dict)
legal_meta    = df_legal.set_index("chunk_id").to_dict(orient="index")
forms_meta    = df_forms.set_index("chunk_id").to_dict(orient="index")
examples_meta = df_examples.set_index("chunk_id").to_dict(orient="index")
print("Metadata lookups built.")

Loading chunk files...
  legal=213,487  forms=10  examples=150
✅ 'article' column present in legal_chunks
[legal] Loading from cache: /Users/capkimkhanh/Documents/DUT/QLDA/RagDraftingAI/RAG/dataset/bm25/bm25_legal_v2.pkl
  Loaded 213,487 docs in 3.2s
[forms] Loading from cache: /Users/capkimkhanh/Documents/DUT/QLDA/RagDraftingAI/RAG/dataset/bm25/bm25_forms_v2.pkl
  Loaded 10 docs in 0.0s
[examples] Loading from cache: /Users/capkimkhanh/Documents/DUT/QLDA/RagDraftingAI/RAG/dataset/bm25/bm25_examples_v2.pkl
  Loaded 150 docs in 0.0s

✅ BM25 indexes ready.
Metadata lookups built.


## 6. FIX 1 — Build ChromaDB ID Maps

In [9]:
print("Building ChromaDB ID maps...")
t0 = time.time()

bm25_to_chroma_legal    = _build_chroma_id_map(df_legal)
bm25_to_chroma_forms    = _build_chroma_id_map(df_forms)
bm25_to_chroma_examples = _build_chroma_id_map(df_examples)

print(f"  legal   : {len(bm25_to_chroma_legal):,} entries")
print(f"  forms   : {len(bm25_to_chroma_forms)} entries")
print(f"  examples: {len(bm25_to_chroma_examples)} entries")
print(f"  Done in {time.time()-t0:.2f}s")

# Spot-check: sample 3 entries
print("\nSample mappings:")
for pk, cv in list(bm25_to_chroma_legal.items())[:3]:
    print(f"  parquet: {pk[:60]}")
    print(f"  chroma : {cv}")
    print()

Building ChromaDB ID maps...
  legal   : 213,487 entries
  forms   : 10 entries
  examples: 150 entries
  Done in 1.73s

Sample mappings:
  parquet: 01/2014/NQLT/CP-UBTƯMTTQVN__Điều_1._Phạm_vi_điều_chỉnh__6c16
  chroma : 9d6e90675d94f79b9825bb8b

  parquet: 01/2014/NQLT/CP-UBTƯMTTQVN__Điều_2._Nguyên_tắc_phối_hợp__ca1
  chroma : 61839530ac8425122795aaa0

  parquet: 01/2014/NQLT/CP-UBTƯMTTQVN__Điều_3._Hướng_dẫn_và_tổ_chức_thự
  chroma : 0279b786373af131e148dd67



## 7. Chunk Expand Index

Build `(doc_id, article) → sorted [(chunk_index, text)]` để ghép full điều luật sau retrieve.

In [10]:
print("Building expand index...")
t0 = time.time()

_expand_index: Dict[Tuple[str, str], List[Tuple[int, str]]] = {}
for row in df_legal.itertuples(index=False):
    key = (row.doc_id, row.article)
    if key not in _expand_index:
        _expand_index[key] = []
    _expand_index[key].append((row.chunk_index, row.text))
for key in _expand_index:
    _expand_index[key].sort(key=lambda x: x[0])

print(f"✅ {len(_expand_index):,} (doc_id, article) pairs | {time.time()-t0:.1f}s")
multi_chunk = sum(1 for v in _expand_index.values() if len(v) > 1)
print(f"   Điều có >1 chunk: {multi_chunk:,} ({multi_chunk/len(_expand_index)*100:.1f}%)")

Building expand index...
✅ 152,557 (doc_id, article) pairs | 2.0s
   Điều có >1 chunk: 27,862 (18.3%)


## 8. FIX 4 — Expand Helper với Word-Count Guard

In [11]:
def expand_legal_chunk(
    metadata: Dict,
    fallback_text: str = "",
    max_chunks: int = MAX_EXPAND_CHUNKS,
    max_words: int = MAX_EXPAND_WORDS,
) -> Tuple[str, int]:
    """
    Ghép full text điều luật từ expand index.

    FIX 3: guard khi 'article' vắng trong metadata → warning + fallback.
    FIX 4: truncate khi tổng từ vượt max_words → tránh overflow context window.

    Returns:
        (expanded_text, n_chunks_used)
    """
    doc_id  = metadata.get("doc_id", "")
    article = metadata.get("article", "")

    # FIX 3: guard thiếu article
    if not article:
        warnings.warn(
            f"metadata thiếu 'article' cho doc_id='{doc_id}'. "
            "Expand fallback về text gốc. Kiểm tra LEGAL_META_COLS khi re-embed.",
            RuntimeWarning, stacklevel=2,
        )
        return fallback_text, 1

    key = (doc_id, article)
    if key not in _expand_index:
        return fallback_text, 1

    merged_parts: List[str] = []
    total_words = 0
    n_used = 0

    for idx, (_, text) in enumerate(_expand_index[key][:max_chunks]):
        # Bỏ header lặp ở chunk 2+ (dạng "[Điều X. ...]")
        if idx > 0:
            lines = text.split("\n")
            if lines and lines[0].startswith("[") and article in lines[0]:
                text = "\n".join(lines[1:]).strip()
        if not text:
            continue

        # FIX 4: word-count guard
        chunk_words = len(text.split())
        if total_words + chunk_words > max_words:
            remaining = max_words - total_words
            if remaining > 50:  # chỉ thêm nếu còn đủ nghĩa
                truncated = " ".join(text.split()[:remaining])
                merged_parts.append(truncated + " [...]")
                n_used += 1
            break

        merged_parts.append(text)
        total_words += chunk_words
        n_used += 1

    if not merged_parts:
        return fallback_text, 1

    return "\n".join(merged_parts), n_used


# Test expand
test_meta = {"doc_id": df_legal.iloc[0]["doc_id"], "article": df_legal.iloc[0]["article"]}
expanded, n = expand_legal_chunk(test_meta)
total_chunks = len(_expand_index.get((test_meta["doc_id"], test_meta["article"]), []))
print(f"Test expand: {total_chunks} chunk(s) available → used {n} → {len(expanded.split())} từ")

# Test FIX 3: thiếu article
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    result, n = expand_legal_chunk({"doc_id": "test", "article": ""}, fallback_text="FALLBACK")
    assert result == "FALLBACK"
    assert len(w) == 1 and "article" in str(w[0].message)
print("✅ FIX 3 guard: missing article → fallback + warning (OK)")

Test expand: 1 chunk(s) available → used 1 → 54 từ
✅ FIX 3 guard: missing article → fallback + warning (OK)


## 9. Load Embedding Model & ChromaDB

In [12]:
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
t0 = time.time()
MODEL_EMBED.mkdir(parents=True, exist_ok=True)

try:
    # Prefer local cache first
    embed_model = SentenceTransformer(
        EMBED_MODEL_NAME, device=DEVICE, cache_folder=str(MODEL_EMBED), local_files_only=True
    )
    load_source = "local cache"
except OSError:
    # Fallback: download from Hugging Face if not cached
    print("Local cache not found. Downloading model from Hugging Face...")
    embed_model = SentenceTransformer(
        EMBED_MODEL_NAME, device=DEVICE, cache_folder=str(MODEL_EMBED), local_files_only=False
    )
    load_source = "huggingface hub"

try:
    EMBED_DIM = embed_model.get_embedding_dimension()
except AttributeError:
    EMBED_DIM = embed_model.get_sentence_embedding_dimension()

print(f"✅ Loaded in {time.time()-t0:.1f}s from {load_source} | dim={EMBED_DIM} | device={DEVICE}")

Loading embedding model: intfloat/multilingual-e5-large-instruct


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Loaded in 1.2s from local cache | dim=1024 | device=cpu


In [13]:
print(f"Connecting to ChromaDB: {CHROMA_DIR}")
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)
col_legal_all = chroma_client.get_collection("legal_chunks")
col_forms     = chroma_client.get_collection("forms_chunks")
col_examples  = chroma_client.get_collection("examples_chunks")

print(f"✅ legal_chunks    : {col_legal_all.count():,} docs")
print(f"✅ forms_chunks    : {col_forms.count():,} docs")
print(f"✅ examples_chunks : {col_examples.count():,} docs")

# FIX 3: Spot-check 'article' có trong ChromaDB metadata
_sample = col_legal_all.get(limit=1, include=["metadatas"])
if _sample["metadatas"] and "article" in _sample["metadatas"][0]:
    print("✅ FIX 3: 'article' field present in ChromaDB metadata")
else:
    print("⚠️  FIX 3 WARNING: 'article' NOT found in ChromaDB metadata!")
    print("   Expand sẽ luôn fallback. Cần re-embed với LEGAL_META_COLS bao gồm 'article'.")
    print(f"   Sample metadata keys: {list(_sample['metadatas'][0].keys()) if _sample['metadatas'] else 'N/A'}")

Connecting to ChromaDB: /Users/capkimkhanh/Documents/DUT/QLDA/RagDraftingAI/RAG/dataset/chromadb
✅ legal_chunks    : 213,487 docs
✅ forms_chunks    : 10 docs
✅ examples_chunks : 150 docs
✅ FIX 3: 'article' field present in ChromaDB metadata


## 10. Load Reranker

In [14]:
print(f"Loading reranker: {RERANK_MODEL_NAME}")
MODEL_RERANK.mkdir(parents=True, exist_ok=True)
t0 = time.time()

try:
    reranker = CrossEncoder(
        RERANK_MODEL_NAME,
        device=DEVICE,
        cache_folder=str(MODEL_RERANK),
        max_length=512,
        local_files_only=True,
    )
    load_source = "local cache"
except OSError:
    print("Local cache not found. Downloading reranker from Hugging Face...")
    reranker = CrossEncoder(
        RERANK_MODEL_NAME,
        device=DEVICE,
        cache_folder=str(MODEL_RERANK),
        max_length=512,
        local_files_only=False,
    )
    load_source = "huggingface hub"

print(f"✅ Reranker loaded in {time.time()-t0:.1f}s from {load_source}")

Loading reranker: BAAI/bge-reranker-v2-m3
Local cache not found. Downloading reranker from Hugging Face...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

✅ Reranker loaded in 35.9s from huggingface hub


## 11. HybridRetrieverV4 Class

**FIX 1 core:** `_bm25_search()` giờ trả về `chroma_id` thay vì `parquet chunk_id`,
nên `_rrf()` merge đúng không gian ID với dense results.

In [15]:
class HybridRetrieverV4:
    """
    BM25 bigram + ChromaDB dense + RRF (ID-corrected) + CrossEncoder + dedup.

    FIX 1: BM25 parquet chunk_ids được convert sang chroma_ids trước RRF.
    """

    def __init__(
        self,
        bm25_index: BM25Okapi,
        bm25_ids: List[str],               # parquet chunk_ids
        bm25_to_chroma: Dict[str, str],    # FIX 1: parquet chunk_id → chroma_id
        chroma_col,
        meta_lookup: Dict,
        embed_model: SentenceTransformer,
        reranker: CrossEncoder,
        instruction: str,
        collection_name: str,
        bm25_top_k: int         = BM25_TOP_K,
        dense_top_k: int        = DENSE_TOP_K,
        rrf_k: int              = RRF_K,
        rerank_top_n: int       = RERANK_TOP_N,
        final_top_k: int        = FINAL_TOP_K,
        max_chunks_per_doc: int = MAX_CHUNKS_PER_DOC,
    ):
        self.bm25               = bm25_index
        self.bm25_ids           = bm25_ids
        self.bm25_to_chroma     = bm25_to_chroma
        self.col                = chroma_col
        self.meta               = meta_lookup
        self.embed_model        = embed_model
        self.reranker           = reranker
        self.instruction        = instruction
        self.name               = collection_name
        self.bm25_top_k         = bm25_top_k
        self.dense_top_k        = dense_top_k
        self.rrf_k              = rrf_k
        self.rerank_top_n       = rerank_top_n
        self.final_top_k        = final_top_k
        self.max_chunks_per_doc = max_chunks_per_doc
        # Reverse map cho fallback metadata lookup
        self._chroma_to_parquet = {v: k for k, v in bm25_to_chroma.items()}

    def _bm25_search(self, query: str) -> List[Tuple[str, float]]:
        """
        FIX 1: Convert parquet chunk_id → chroma_id trước khi trả về.
        Trả về List[(chroma_id, bm25_score)].
        """
        tokens = tokenize_for_bm25(query)
        if not tokens:
            return []
        scores  = self.bm25.get_scores(tokens)
        top_idx = np.argsort(scores)[::-1][: self.bm25_top_k]
        results = []
        for i in top_idx:
            if scores[i] <= 0:
                continue
            parquet_id = self.bm25_ids[i]
            chroma_id  = self.bm25_to_chroma.get(parquet_id)
            if chroma_id is None:
                continue  # skip unmapped IDs (không xảy ra nếu map được build đúng)
            results.append((chroma_id, float(scores[i])))
        return results

    def _dense_search(
        self, query: str, where: Optional[Dict] = None
    ) -> List[Tuple[str, float, str, Dict]]:
        """Trả về List[(chroma_id, distance, doc_text, metadata)]."""
        q_vec = self.embed_model.encode(
            [self.instruction + query],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )[0]
        kwargs: Dict = dict(
            query_embeddings=[q_vec.tolist()],
            n_results=self.dense_top_k,
            include=["documents", "metadatas", "distances"],
        )
        if where:
            kwargs["where"] = where
        res = self.col.query(**kwargs)
        return [
            (cid, float(dist), doc, meta or {})
            for cid, dist, doc, meta in zip(
                res["ids"][0], res["distances"][0],
                res["documents"][0], res["metadatas"][0],
            )
        ]

    def _rrf(
        self,
        bm25_results: List[Tuple[str, float]],
        dense_results: List[Tuple[str, float, str, Dict]],
    ) -> List[Tuple[str, float]]:
        """
        FIX 1: Cả hai danh sách đều dùng chroma_id → overlap được detect đúng.
        Chunk xuất hiện ở cả BM25 lẫn dense sẽ được boost.
        """
        scores: Dict[str, float] = {}
        for rank, (cid, _) in enumerate(bm25_results, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (self.rrf_k + rank)
        for rank, (cid, _, _, _) in enumerate(dense_results, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (self.rrf_k + rank)
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)

    def _rerank(
        self, query: str, candidate_ids: List[str], doc_texts: Dict[str, str]
    ) -> List[Tuple[str, float]]:
        pairs = [(query, doc_texts[cid]) for cid in candidate_ids if cid in doc_texts]
        if not pairs:
            return [(cid, 0.0) for cid in candidate_ids]
        scores = self.reranker.predict(pairs, show_progress_bar=False)
        return sorted(
            zip(candidate_ids[: len(pairs)], scores.tolist()),
            key=lambda x: x[1], reverse=True,
        )

    def _dedup_by_doc(
        self, ranked: List[Tuple[str, float]], chroma_meta: Dict[str, Dict]
    ) -> List[Tuple[str, float]]:
        seen: Dict[str, int] = {}
        result = []
        for cid, score in ranked:
            doc_id = chroma_meta.get(cid, {}).get("doc_id", cid)
            if seen.get(doc_id, 0) < self.max_chunks_per_doc:
                result.append((cid, score))
                seen[doc_id] = seen.get(doc_id, 0) + 1
        return result

    def retrieve(
        self,
        query: str,
        type_filter: Optional[str] = None,
        form_candidates: Optional[List[str]] = None,
        use_reranker: bool = True,
    ) -> List[Dict]:
        t0 = time.time()

        where: Optional[Dict] = None
        if type_filter:
            where = {"type_normalized": type_filter}

        if form_candidates:
            where = (
                {"form_id": form_candidates[0]}
                if len(form_candidates) == 1
                else {"form_id": {"$in": form_candidates}}
            )
            bm25_res  = []
            dense_res = self._dense_search(query, where=where)
        else:
            bm25_res  = self._bm25_search(query)   # → chroma_ids (FIX 1)
            dense_res = self._dense_search(query, where=where)

        # Build lookups (keyed by chroma_id)
        doc_texts:   Dict[str, str]  = {}
        chroma_meta: Dict[str, Dict] = {}
        for cid, _, doc, meta in dense_res:
            doc_texts[cid]   = doc
            chroma_meta[cid] = meta

        # BM25-only hits: lookup text từ parquet via reverse map
        for cid, _ in bm25_res:
            if cid not in doc_texts:
                parquet_id = self._chroma_to_parquet.get(cid)
                if parquet_id and parquet_id in self.meta:
                    doc_texts[cid]   = self.meta[parquet_id].get("text", "")
                    chroma_meta[cid] = self.meta[parquet_id]

        bm25_score_map = {cid: s for cid, s in bm25_res}
        dense_dist_map = {cid: d for cid, d, _, _ in dense_res}

        rrf_ranked    = self._rrf(bm25_res, dense_res)
        top_n_ids     = [cid for cid, _ in rrf_ranked[: self.rerank_top_n]]
        rrf_score_map = {cid: s for cid, s in rrf_ranked}

        if use_reranker and top_n_ids:
            final_ranked = self._rerank(query, top_n_ids, doc_texts)
        else:
            final_ranked = [(cid, rrf_score_map.get(cid, 0.0)) for cid in top_n_ids]

        deduped = self._dedup_by_doc(final_ranked, chroma_meta)

        results = []
        for cid, rerank_score in deduped[: self.final_top_k]:
            meta = chroma_meta.get(cid, {})
            results.append({
                "chunk_id"    : cid,
                "text"        : doc_texts.get(cid, ""),
                "metadata"    : meta,
                "bm25_score"  : round(bm25_score_map.get(cid, 0.0), 4),
                "dense_dist"  : round(dense_dist_map.get(cid, 1.0), 4),
                "rrf_score"   : round(rrf_score_map.get(cid, 0.0), 6),
                "rerank_score": round(float(rerank_score), 4),
                "latency_ms"  : round((time.time() - t0) * 1000, 1),
                "in_both"     : cid in bm25_score_map and cid in dense_dist_map,
            })
        return results


print("HybridRetrieverV4 defined.")

HybridRetrieverV4 defined.


## 12. Khởi tạo Retrievers

In [16]:
retriever_legal = HybridRetrieverV4(
    bm25_index=bm25_legal, bm25_ids=ids_legal,
    bm25_to_chroma=bm25_to_chroma_legal,
    chroma_col=col_legal_all, meta_lookup=legal_meta,
    embed_model=embed_model, reranker=reranker,
    instruction=INSTRUCTIONS["legal"], collection_name="legal",
)

retriever_forms = HybridRetrieverV4(
    bm25_index=bm25_forms, bm25_ids=ids_forms,
    bm25_to_chroma=bm25_to_chroma_forms,
    chroma_col=col_forms, meta_lookup=forms_meta,
    embed_model=embed_model, reranker=reranker,
    instruction=INSTRUCTIONS["forms"], collection_name="forms",
    final_top_k=1, max_chunks_per_doc=1,
)

retriever_examples = HybridRetrieverV4(
    bm25_index=bm25_examples, bm25_ids=ids_examples,
    bm25_to_chroma=bm25_to_chroma_examples,
    chroma_col=col_examples, meta_lookup=examples_meta,
    embed_model=embed_model, reranker=reranker,
    instruction=INSTRUCTIONS["examples"], collection_name="examples",
    final_top_k=1, max_chunks_per_doc=1,
)

print("✅ All retrievers v4 ready.")

✅ All retrievers v4 ready.


## 13. Public API

**FIX 2** nằm trong `retrieve_forms()`: 1-candidate path dùng `col.query()` thay vì `col.get()`.

In [17]:
def retrieve_legal(
    query: str,
    top_k: int = FINAL_TOP_K,
    type_filter: Optional[str] = None,
    use_reranker: bool = True,
    expand: bool = True,
) -> List[Dict]:
    """
    Retrieve điều khoản pháp luật liên quan.
    expand=True: ghép full text điều luật với word-count guard (FIX 4).
    type_filter: "LUẬT" | "NGHỊ ĐỊNH" | "NGHỊ QUYẾT" | "PHÁP LỆNH"
    """
    retriever_legal.final_top_k = top_k
    results = retriever_legal.retrieve(
        query, type_filter=type_filter, use_reranker=use_reranker
    )
    if not expand:
        return results
    for r in results:
        expanded_text, n_used = expand_legal_chunk(
            metadata=r["metadata"],
            fallback_text=r["text"],
        )
        r["text"]             = expanded_text
        r["n_chunks_expanded"] = n_used
        r["expanded_words"]   = len(expanded_text.split())
    return results


def retrieve_forms(
    query: str,
    use_reranker: bool = True,
) -> List[Dict]:
    """
    Retrieve biểu mẫu hành chính phù hợp nhất.
    FIX 2: 1-candidate path dùng dense query thay vì col.get() ngẫu nhiên.
    """
    form_candidates = detect_form_candidates(query)

    if not form_candidates:
        return retriever_forms.retrieve(query, use_reranker=use_reranker)

    if len(form_candidates) == 1:
        # FIX 2: dùng query() để lấy chunk semantically relevant nhất
        q_vec = embed_model.encode(
            [INSTRUCTIONS["forms"] + query],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )[0]
        res = col_forms.query(
            query_embeddings=[q_vec.tolist()],
            n_results=1,
            where={"form_id": form_candidates[0]},
            include=["documents", "metadatas", "distances"],
        )
        if res["ids"] and res["ids"][0]:
            return [{
                "chunk_id"    : res["ids"][0][0],
                "text"        : res["documents"][0][0],
                "metadata"    : res["metadatas"][0][0],
                "dense_dist"  : round(float(res["distances"][0][0]), 4),
                "rerank_score": 1.0,
                "source"      : "keyword_match",
            }]
        return retriever_forms.retrieve(query, use_reranker=use_reranker)

    # Nhiều candidates → fetch all rồi rerank
    res = col_forms.get(
        where={"form_id": {"$in": form_candidates}},
        include=["documents", "metadatas"],
    )
    if not res["ids"]:
        return retriever_forms.retrieve(query, use_reranker=use_reranker)

    candidates_list = [
        {"chunk_id": cid, "text": doc, "metadata": meta, "source": "keyword_match"}
        for cid, doc, meta in zip(res["ids"], res["documents"], res["metadatas"])
    ]
    if use_reranker and len(candidates_list) > 1:
        scores = reranker.predict([(query, c["text"]) for c in candidates_list])
        for c, s in zip(candidates_list, scores):
            c["rerank_score"] = float(s)
        candidates_list.sort(key=lambda x: x["rerank_score"], reverse=True)
    else:
        for c in candidates_list:
            c["rerank_score"] = 1.0
    return [candidates_list[0]]


def retrieve_examples(
    query: str,
    top_k: int = 3,
    form_id: Optional[str] = None,
    use_reranker: bool = True,
) -> List[Dict]:
    """Retrieve ví dụ văn bản (few-shot). Ưu tiên: form_id > keyword > dense."""
    retriever_examples.final_top_k = top_k
    if not form_id:
        _candidates = detect_form_candidates(query)
        form_id = _candidates[0] if _candidates else None
    return retriever_examples.retrieve(
        query,
        form_candidates=[form_id] if form_id else None,
        use_reranker=use_reranker,
    )


def retrieve_all(
    query: str,
    legal_top_k: int = FINAL_TOP_K,
    examples_top_k: int = 3,
    legal_type_filter: Optional[str] = None,
    expand_legal: bool = True,
) -> Dict[str, List[Dict]]:
    """Entry point chính cho RAG generation pipeline."""
    form_results = retrieve_forms(query)
    detected_form_id = (
        form_results[0]["metadata"].get("form_id") if form_results else None
    )
    return {
        "legal"   : retrieve_legal(
            query, top_k=legal_top_k,
            type_filter=legal_type_filter,
            expand=expand_legal,
        ),
        "form"    : form_results,
        "examples": retrieve_examples(
            query, top_k=examples_top_k,
            form_id=detected_form_id,
        ),
    }


print("✅ Public API defined.")

✅ Public API defined.


## 14. Test & Benchmark

In [18]:
def print_legal_results(results: List[Dict], query: str):
    print(f"\n{'='*70}")
    print(f"[LEGAL] {query}")
    print(f"{'='*70}")
    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        print(f"\n  [{i}] {meta.get('source_doc_no','?')} | {meta.get('type_normalized','')}")
        print(f"      📌 {meta.get('article','')}")
        print(f"      🎯 rerank={r['rerank_score']:.4f} | bm25={r['bm25_score']:.2f} | "
              f"dense_dist={r['dense_dist']:.4f} | in_both={r.get('in_both',False)}")
        n = r.get('n_chunks_expanded', 1)
        w = r.get('expanded_words', len(r['text'].split()))
        print(f"      📦 expanded: {n} chunk(s) → {w} từ")
        print(f"      📝 {r['text'][:250].replace(chr(10),' ')}...")
    if results:
        print(f"\n  ⏱  Latency: {results[0]['latency_ms']:.0f}ms")


def print_forms_results(results, query):
    print(f"\n{'='*70}")
    print(f"[FORMS] {query}")
    print(f"{'='*70}")
    for r in results:
        meta = r["metadata"]
        print(f"  {meta.get('form_id','?')} | {meta.get('form_type','')} | "
              f"rerank={r.get('rerank_score',0):.4f} | source={r.get('source','pipeline')}")
        print(f"  {meta.get('purpose','')}")


def print_examples_results(results, query):
    print(f"\n{'='*70}")
    print(f"[EXAMPLES] {query}")
    print(f"{'='*70}")
    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        print(f"  [{i}] {meta.get('form_id','?')} | rerank={r.get('rerank_score',0):.4f}")
        print(f"       {meta.get('scenario','')[:120]}")


print("Display helpers defined.")

Display helpers defined.


In [19]:
# ── Test legal + expand ──────────────────────────────────────
legal_queries = [
    "Quy định về hòa giải ở cơ sở",
    "trách nhiệm của Bộ trưởng trong ban hành văn bản",
    "quy trình bổ nhiệm cán bộ công chức",
    "điều kiện thành lập doanh nghiệp",
    "bảo hiểm xã hội bắt buộc cho người lao động",
]

for query in legal_queries:
    results = retrieve_legal(query, expand=True)
    print_legal_results(results, query)
    print("-" * 70)


[LEGAL] Quy định về hòa giải ở cơ sở

  [1] 35/2013/QH13 | LUẬT
      📌 Điều 28. Trách nhiệm quản lý nhà nước về hòa giải ở cơ sở
      🎯 rerank=0.9965 | bm25=84.21 | dense_dist=0.0704 | in_both=True
      📦 expanded: 1 chunk(s) → 233 từ
      📝 Điều 28. Trách nhiệm quản lý nhà nước về hòa giải ở cơ sở 1. Chính phủ thống nhất quản lý nhà nước về hòa giải ở cơ sở. 2. Bộ Tư pháp chịu trách nhiệm trước Chính phủ thực hiện quản lý nhà nước về hòa giải ở cơ sở, có trách nhiệm sau đây: a) Xây dựng...

  [2] 15/2014/NĐ-CP | NGHỊ ĐỊNH
      📌 Điều 1. Phạm vi điều chỉnh
      🎯 rerank=0.9952 | bm25=71.33 | dense_dist=0.0693 | in_both=True
      📦 expanded: 1 chunk(s) → 49 từ
      📝 Điều 1. Phạm vi điều chỉnh Nghị định này quy định chi tiết về phạm vi hòa giải ở cơ sở; hỗ trợ kinh phí cho công tác hòa giải ở cơ sở, hòa giải viên và một số biện pháp thi hành Luật hòa giải ở cơ sở....

  [3] 35/2013/QH13 | LUẬT
      📌 Điều 3. Phạm vi hòa giải ở cơ sở
      🎯 rerank=0.9951 | bm25=68.05 | dense_d

In [20]:
# ── Test forms & examples ────────────────────────────────────
form_queries = [
    "soạn thảo công văn gửi sở tài chính",
    "làm tờ trình đề nghị phê duyệt dự án",
    "viết giấy mời họp ban lãnh đạo",
    "quyết định bổ nhiệm trưởng phòng",
    "giấy giới thiệu cán bộ đi công tác",
    "công điện khẩn ứng phó thiên tai",
]

for query in form_queries:
    res_forms    = retrieve_forms(query)
    res_examples = retrieve_examples(query)
    print_forms_results(res_forms, query)
    print_examples_results(res_examples, query)


[FORMS] soạn thảo công văn gửi sở tài chính
  Form_05 | Công văn | rerank=1.0000 | source=keyword_match
  Giao dịch, trao đổi công việc hành chính

[EXAMPLES] soạn thảo công văn gửi sở tài chính
  [1] Form_05 | rerank=0.1039
       Cục Thuế tỉnh Nghệ An đề nghị hộ kinh doanh triển khai sử dụng hóa đơn điện tử khởi tạo từ máy tính tiền trong năm 2025.
  [2] Form_05 | rerank=0.0385
       Sở Tài nguyên và Môi trường Bắc Ninh yêu cầu các khu công nghiệp tăng cường quan trắc nước thải tự động năm 2023.
  [3] Form_05 | rerank=0.0255
       Trường Đại học Cần Thơ gửi công văn đề nghị các khoa cập nhật minh chứng kiểm định chất lượng chương trình đào tạo năm 2

[FORMS] làm tờ trình đề nghị phê duyệt dự án
  Form_04 | Chỉ thị, Quy chế, Quy định, Thông cáo, Thông báo, Hướng dẫn, Chương trình, Kế hoạch, Phương án, Đề án, Dự án, Báo cáo, Tờ trình, Giấy ủy quyền, Phiếu gửi, Phiếu chuyển, Phiếu báo | rerank=1.0000 | source=keyword_match
  Văn bản hành chính có tên loại chung

[EXAMPLES] làm tờ trì

In [21]:
# ── Demo retrieve_all ────────────────────────────────────────
demo_query = "soạn thảo quyết định bổ nhiệm cán bộ"
print(f"Demo retrieve_all(): '{demo_query}'\n")

t0 = time.time()
all_results = retrieve_all(demo_query)
elapsed = (time.time() - t0) * 1000

print(f"  Legal    : {len(all_results['legal'])} điều luật")
print(f"  Form     : {len(all_results['form'])} form")
print(f"  Examples : {len(all_results['examples'])} ví dụ")
print(f"  Total latency: {elapsed:.0f}ms")

if all_results["legal"]:
    r = all_results["legal"][0]
    print(f"\nTop legal : {r['metadata'].get('source_doc_no','?')} | {r['metadata'].get('article','')}")
    print(f"  rerank={r['rerank_score']:.4f} | {r.get('n_chunks_expanded',1)} chunk(s) | "
          f"{r.get('expanded_words',0)} từ | in_both={r.get('in_both',False)}")

if all_results["form"]:
    r = all_results["form"][0]
    print(f"Top form  : {r['metadata'].get('form_id','?')} | {r['metadata'].get('form_type','')} | "
          f"source={r.get('source','pipeline')}")

if all_results["examples"]:
    r = all_results["examples"][0]
    print(f"Top example: {r['metadata'].get('form_id','?')} | rerank={r.get('rerank_score',0):.4f}")
    print(f"  {r['metadata'].get('scenario','')[:100]}")

Demo retrieve_all(): 'soạn thảo quyết định bổ nhiệm cán bộ'

  Legal    : 3 điều luật
  Form     : 1 form
  Examples : 3 ví dụ
  Total latency: 4052ms

Top legal : 43/2009/QH12 | Điều 29. Thẩm quyền bổ nhiệm cán bộ Ban chỉ huy quân sự bộ, ngành trung ương, cán bộ chỉ huy quân sự cơ sở và cán bộ dân quân tự vệ
  rerank=0.8256 | 1 chunk(s) | 358 từ | in_both=False
Top form  : Form_02 | Quyết định | source=keyword_match
Top example: Form_02 | rerank=0.8493
  Bộ Tài chính bổ nhiệm Vụ trưởng Vụ Chính sách Thuế.


---
## Summary

| Component | v3 | v4 |
|---|---|---|
| **RRF ID space** | Mixed (parquet vs SHA-256) → no boost | Unified chroma_id → **correct boost** |
| **Forms 1-candidate** | `col.get()` → random chunk | `col.query()` → semantic-relevant chunk |
| **Expand guard** | No article check | Assert + warning + fallback khi thiếu `article` |
| **Expand word limit** | Chunk-count only (max 6) | Word-count guard (`MAX_EXPAND_WORDS=2400`) |
| **BM25_TOP_K** | 30 | 30 |
| **DENSE_TOP_K** | 30 | 30 |
| **RERANK_TOP_N** | 10 | 10 |
| **FINAL_TOP_K** | 3 | 3 |

**Public API (không đổi — backward compatible):**
```python
from hybrid_retrieval_v4 import init_retriever, retrieve_all
init_retriever()           # gọi 1 lần
results = retrieve_all(query)
```